In [2]:
# STEP 1: Load the external knowledgebase and convert into pages.
# Example - Knowledgebase in pdf format
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("Artificial-Intelligence.pdf")
pdf_pages = loader.load()
print(f'No of pages: {len(pdf_pages)}')

C:\Users\BG108836\AppData\Local\Temp\ipykernel_28980\3714349099.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


No of pages: 80


In [3]:
# STEP 2: Split pages into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Spilit pages into small chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ",", " "]
)
split_docs = splitter.split_documents(pdf_pages)
print(f'Total no of chunks {len(split_docs)}')

Total no of chunks 344


In [5]:
# STEP 3: Create vector embeddings using FAISS

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(split_docs, embedding=embedding_model)
vector_store.save_local("faiss_ai_book")
print('AI book vector store is created using FAISS...')



AI book vector store is created using FAISS...


In [ ]:
# STEP 3: Create vector embeddings using Chroma
from langchain_community.vectorstores import Chroma

persist_directory = "chroma_ai_book"

vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print('python book vector store is created using Chroma...')

In [ ]:
# STEP 4: Build Retrieval Chain

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print("Model is ready")

USER_PROMPT_TEMPLATE = """Use the following pieces of the context to answer user's question.
If you don't know the answer, just say you don't know, don't try to make up the answer.
------------------------
{context}
Question: {question}
"""

USER_PROMPT = PromptTemplate(
    template=USER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
retriever = vector_store.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    model,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": USER_PROMPT}
)
print(qa_chain)
print('Retrieval chain is built successfully...')

In [ ]:
# STEP 5: Build UI interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    print(f"Source documents: {source_documents}")

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Python Chatbot",
    description="Ask questions about AI programming."
)

# Launch the interface
iface.launch(share=True, server_port=7861)